# AMPS I · Week 09 — HW8: Nonlinear MMS with PETSc SNES
**Due:** before the Week 10 lecture · **Points:** 100 · **Submit:** this notebook (`.ipynb`) with all outputs

**Rules**
- Before submitting: *Runtime → Restart session and run all*. Notebooks that do not run end to end receive 0.
- An error inside a `%%fenicsx` cell does **not** stop *Run all*. Read every output before you submit.
- LLM assistants are encouraged. Fill in the **AI-usage log** at the end.
- Every number you report must come from a cell you ran. **After every `solve()` print the converged reason** — a table built on an unconverged
  solve receives no points.

| Question | Topic | Points |
|---|---|:---:|
| Q1 | Kirchhoff transform · Jacobian · exact 1D Bratu | 20 |
| Q2 | $-\nabla\cdot(e^u\nabla u)=f$: SNES, MMS, Kirchhoff cross-check, stopping tolerance | 40 |
| Q3 | 1D Bratu: continuation, fold, exact $\lambda^*$ | 20 |
| Q4 | LLM debugging: a continuation script with four bugs | 20 |

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## Q1 — Theory (20 pts)

**(a) (6) Kirchhoff transform.** Let $q\in C^1(\mathbb R)$ with $q(s)\ge q_0>0$, and $K(u)=\int_0^uq(s)\,ds$. Show that $u\in H^1(\Omega)$ solves
$-\nabla\cdot(q(u)\nabla u)=f$ in $\Omega$, $u=g$ on $\partial\Omega$ (weakly) if and only if $w=K(u)$ solves $-\Delta w=f$, $w=K(g)$ on $\partial\Omega$.
Conclude that the nonlinear problem has exactly one solution. Write $K$ and $K^{-1}$ for $q(u)=e^u$ (the conductivity of Q2) — is $q\ge q_0>0$
satisfied there, and does it matter for the argument?

**(b) (4) Jacobian.** For $F(u;v)=\int_\Omega e^u\nabla u\cdot\nabla v\,dx-\int_\Omega fv\,dx$ compute $J(u;w,v)$. Which term would Picard iteration drop?

**(c) (10) The one-dimensional Bratu problem.** $u''+\lambda e^u=0$ on $(0,1)$, $u(0)=u(1)=0$, $\lambda>0$.

1. Show that $u(x)=-2\ln\!\Bigl[\dfrac{\cosh\bigl((x-\tfrac12)\tfrac\theta2\bigr)}{\cosh(\theta/4)}\Bigr]$ satisfies the boundary conditions, and that it solves
   the equation if and only if $\theta=\sqrt{2\lambda}\,\cosh(\theta/4)$.
2. Hence $\lambda(\theta)=\dfrac{\theta^2}{2\cosh^2(\theta/4)}$. Show that the maximum $\lambda^*$ satisfies $\theta\tanh(\theta/4)=4$ and compute $\theta^*$, $\lambda^*$ and
   $u^*(\tfrac12)$ to 8 digits with NumPy (next cell; no `%%fenicsx`). How many solutions are there for $\lambda<\lambda^*$, $=\lambda^*$, $>\lambda^*$?
3. Compute $u(\tfrac12)$ of **both** solutions at $\lambda=2$.
4. Frank-Kamenetskii writes the slab problem on $(-1,1)$ as $u''+\delta e^u=0$. Show that $\delta=\lambda/4$ and give $\delta_{\rm cr}$.

> *your answer to (a)–(c)*

In [ ]:
# TODO: Q1(c) with NumPy: theta*, lam*, u*(1/2); both solutions at lam = 2

## Q2 — $-\nabla\cdot(e^u\nabla u)=f$ with SNES (40 pts)

Manufactured solution $\tilde u=xy+\sin(\pi x)\sin(\pi y)$ (non-zero boundary data on two sides), $f=-\nabla\cdot(e^{\tilde u}\nabla\tilde u)$ by UFL.
The setup (`hw_problem`) is given in the cells. Use `NonlinearProblem` with LU, `bt`, `snes_rtol` $10^{-10}$, `snes_atol` $10^{-12}$, and the
initial guess $u^0=g$ on $\partial\Omega$, 0 inside (`fem.set_bc(uh.x.array, [bc])`).

**(a) (10)** P1, $N=32$, `-np 2`: print the reason, the $\|F(u^k)\|$ history and $\|F_{k+1}\|/\|F_k\|^2$, and the $L^2$ error. Run the same with `-np 1`
(second cell) and compare the histories. Is the convergence quadratic?

**(b) (12)** P1 and P2, $N=8,16,32,64$: the reason and iteration count for each solve, $L^2$ and $H^1$ errors and rates. Predict the rates first.
Does the iteration count depend on $N$?

**(c) (10) Kirchhoff cross-check.** By Q1(a), $w=e^u-1$ solves $-\Delta w=f$ with $w=e^g-1$ on $\partial\Omega$. Solve this **linear** problem with P$k$
(`LinearProblem`), set $u_K=\ln(1+w_h)$ (a UFL expression) and report $\|u_K-\tilde u\|_{L^2}$ and $\|u_h-u_K\|_{L^2}$ with rates, P1 and P2.
Is $u_K$ the same discrete function as $u_h$? Why or why not — and what does the comparison verify?

**(d) (8) Stopping tolerance.** P2, $N=8,\dots,64$, with `snes_rtol` $=10^{-2},10^{-3},10^{-4},10^{-6},10^{-8}$ (`snes_atol` $10^{-14}$). Report the
iteration counts and the $L^2$ rates. Which tolerances give clean third-order rates up to $N=64$? Explain the failure mode in one or two sentences.

In [ ]:
%%fenicsx -np 2
# TODO: Q2(a), -np 2

from mpi4py import MPI
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem, NonlinearProblem

comm = MPI.COMM_WORLD
LU = {"ksp_type": "preonly", "pc_type": "lu"}
TIGHT = {**LU, "snes_rtol": 1e-10, "snes_atol": 1e-12, "snes_max_it": 50}

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def boundary_dofs(msh, V):
    tdim = msh.topology.dim
    msh.topology.create_connectivity(tdim - 1, tdim)
    return fem.locate_dofs_topological(V, tdim - 1, mesh.exterior_facet_indices(msh.topology))

def norm_L2(msh, e):
    return np.sqrt(msh.comm.allreduce(fem.assemble_scalar(fem.form(e ** 2 * ufl.dx)), op=MPI.SUM))

_n = [0]
def snes_solve(F, uh, bcs, opts):
    """Solve F(uh; v) = 0 with PETSc SNES. Returns (reason, iterations, history of ||F||)."""
    _n[0] += 1
    problem = NonlinearProblem(F, uh, bcs=bcs, petsc_options_prefix=f"nls{_n[0]}_", petsc_options=opts)
    hist = []
    problem.solver.setMonitor(lambda snes, it, rnorm: hist.append(rnorm))
    problem.solve()
    return problem.solver.getConvergedReason(), problem.solver.getIterationNumber(), hist

kappa = lambda u: ufl.exp(u)                             # conductivity k(u) = e^u

def hw_problem(N, k):
    """-div(e^u grad u) = f on the unit square, u = g = u_ex on the boundary."""
    msh = mesh.create_unit_square(comm, N, N)
    V = fem.functionspace(msh, ("Lagrange", k))
    x = ufl.SpatialCoordinate(msh)
    u_ex = x[0] * x[1] + ufl.sin(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1])
    f = -ufl.div(kappa(u_ex) * ufl.grad(u_ex))
    g = fem.Function(V); g.interpolate(fem.Expression(u_ex, V.element.interpolation_points))
    return msh, V, u_ex, f, fem.dirichletbc(g, boundary_dofs(msh, V))

def hw_solve(N, k, opts=TIGHT):
    """SNES from u0 = g on the boundary, 0 inside. Returns msh, uh, u_ex, f, bc, reason, its, history."""
    msh, V, u_ex, f, bc = hw_problem(N, k)
    uh = fem.Function(V); v = ufl.TestFunction(V)
    fem.set_bc(uh.x.array, [bc])
    F = kappa(uh) * ufl.inner(ufl.grad(uh), ufl.grad(v)) * ufl.dx - f * v * ufl.dx
    reason, its, hist = snes_solve(F, uh, [bc], opts)
    return msh, uh, u_ex, f, bc, reason, its, hist

In [ ]:
%%fenicsx -np 1
# TODO: Q2(a), the same with -np 1

> *your answer to (a)*

In [ ]:
%%fenicsx -np 2
# TODO: Q2(b) rates and iteration counts, -np 2

from mpi4py import MPI
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem, NonlinearProblem

comm = MPI.COMM_WORLD
LU = {"ksp_type": "preonly", "pc_type": "lu"}
TIGHT = {**LU, "snes_rtol": 1e-10, "snes_atol": 1e-12, "snes_max_it": 50}

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def boundary_dofs(msh, V):
    tdim = msh.topology.dim
    msh.topology.create_connectivity(tdim - 1, tdim)
    return fem.locate_dofs_topological(V, tdim - 1, mesh.exterior_facet_indices(msh.topology))

def norm_L2(msh, e):
    return np.sqrt(msh.comm.allreduce(fem.assemble_scalar(fem.form(e ** 2 * ufl.dx)), op=MPI.SUM))

_n = [0]
def snes_solve(F, uh, bcs, opts):
    """Solve F(uh; v) = 0 with PETSc SNES. Returns (reason, iterations, history of ||F||)."""
    _n[0] += 1
    problem = NonlinearProblem(F, uh, bcs=bcs, petsc_options_prefix=f"nls{_n[0]}_", petsc_options=opts)
    hist = []
    problem.solver.setMonitor(lambda snes, it, rnorm: hist.append(rnorm))
    problem.solve()
    return problem.solver.getConvergedReason(), problem.solver.getIterationNumber(), hist

kappa = lambda u: ufl.exp(u)                             # conductivity k(u) = e^u

def hw_problem(N, k):
    """-div(e^u grad u) = f on the unit square, u = g = u_ex on the boundary."""
    msh = mesh.create_unit_square(comm, N, N)
    V = fem.functionspace(msh, ("Lagrange", k))
    x = ufl.SpatialCoordinate(msh)
    u_ex = x[0] * x[1] + ufl.sin(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1])
    f = -ufl.div(kappa(u_ex) * ufl.grad(u_ex))
    g = fem.Function(V); g.interpolate(fem.Expression(u_ex, V.element.interpolation_points))
    return msh, V, u_ex, f, fem.dirichletbc(g, boundary_dofs(msh, V))

def hw_solve(N, k, opts=TIGHT):
    """SNES from u0 = g on the boundary, 0 inside. Returns msh, uh, u_ex, f, bc, reason, its, history."""
    msh, V, u_ex, f, bc = hw_problem(N, k)
    uh = fem.Function(V); v = ufl.TestFunction(V)
    fem.set_bc(uh.x.array, [bc])
    F = kappa(uh) * ufl.inner(ufl.grad(uh), ufl.grad(v)) * ufl.dx - f * v * ufl.dx
    reason, its, hist = snes_solve(F, uh, [bc], opts)
    return msh, uh, u_ex, f, bc, reason, its, hist

> *your prediction, table and answer to (b)*

In [ ]:
%%fenicsx -np 2
# TODO: Q2(c) Kirchhoff cross-check, -np 2

from mpi4py import MPI
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem, NonlinearProblem

comm = MPI.COMM_WORLD
LU = {"ksp_type": "preonly", "pc_type": "lu"}
TIGHT = {**LU, "snes_rtol": 1e-10, "snes_atol": 1e-12, "snes_max_it": 50}

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def boundary_dofs(msh, V):
    tdim = msh.topology.dim
    msh.topology.create_connectivity(tdim - 1, tdim)
    return fem.locate_dofs_topological(V, tdim - 1, mesh.exterior_facet_indices(msh.topology))

def norm_L2(msh, e):
    return np.sqrt(msh.comm.allreduce(fem.assemble_scalar(fem.form(e ** 2 * ufl.dx)), op=MPI.SUM))

_n = [0]
def snes_solve(F, uh, bcs, opts):
    """Solve F(uh; v) = 0 with PETSc SNES. Returns (reason, iterations, history of ||F||)."""
    _n[0] += 1
    problem = NonlinearProblem(F, uh, bcs=bcs, petsc_options_prefix=f"nls{_n[0]}_", petsc_options=opts)
    hist = []
    problem.solver.setMonitor(lambda snes, it, rnorm: hist.append(rnorm))
    problem.solve()
    return problem.solver.getConvergedReason(), problem.solver.getIterationNumber(), hist

kappa = lambda u: ufl.exp(u)                             # conductivity k(u) = e^u

def hw_problem(N, k):
    """-div(e^u grad u) = f on the unit square, u = g = u_ex on the boundary."""
    msh = mesh.create_unit_square(comm, N, N)
    V = fem.functionspace(msh, ("Lagrange", k))
    x = ufl.SpatialCoordinate(msh)
    u_ex = x[0] * x[1] + ufl.sin(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1])
    f = -ufl.div(kappa(u_ex) * ufl.grad(u_ex))
    g = fem.Function(V); g.interpolate(fem.Expression(u_ex, V.element.interpolation_points))
    return msh, V, u_ex, f, fem.dirichletbc(g, boundary_dofs(msh, V))

def hw_solve(N, k, opts=TIGHT):
    """SNES from u0 = g on the boundary, 0 inside. Returns msh, uh, u_ex, f, bc, reason, its, history."""
    msh, V, u_ex, f, bc = hw_problem(N, k)
    uh = fem.Function(V); v = ufl.TestFunction(V)
    fem.set_bc(uh.x.array, [bc])
    F = kappa(uh) * ufl.inner(ufl.grad(uh), ufl.grad(v)) * ufl.dx - f * v * ufl.dx
    reason, its, hist = snes_solve(F, uh, [bc], opts)
    return msh, uh, u_ex, f, bc, reason, its, hist

> *your answer to (c)*

In [ ]:
%%fenicsx -np 2
# TODO: Q2(d) stopping tolerance, P2, -np 2

from mpi4py import MPI
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem, NonlinearProblem

comm = MPI.COMM_WORLD
LU = {"ksp_type": "preonly", "pc_type": "lu"}
TIGHT = {**LU, "snes_rtol": 1e-10, "snes_atol": 1e-12, "snes_max_it": 50}

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def boundary_dofs(msh, V):
    tdim = msh.topology.dim
    msh.topology.create_connectivity(tdim - 1, tdim)
    return fem.locate_dofs_topological(V, tdim - 1, mesh.exterior_facet_indices(msh.topology))

def norm_L2(msh, e):
    return np.sqrt(msh.comm.allreduce(fem.assemble_scalar(fem.form(e ** 2 * ufl.dx)), op=MPI.SUM))

_n = [0]
def snes_solve(F, uh, bcs, opts):
    """Solve F(uh; v) = 0 with PETSc SNES. Returns (reason, iterations, history of ||F||)."""
    _n[0] += 1
    problem = NonlinearProblem(F, uh, bcs=bcs, petsc_options_prefix=f"nls{_n[0]}_", petsc_options=opts)
    hist = []
    problem.solver.setMonitor(lambda snes, it, rnorm: hist.append(rnorm))
    problem.solve()
    return problem.solver.getConvergedReason(), problem.solver.getIterationNumber(), hist

kappa = lambda u: ufl.exp(u)                             # conductivity k(u) = e^u

def hw_problem(N, k):
    """-div(e^u grad u) = f on the unit square, u = g = u_ex on the boundary."""
    msh = mesh.create_unit_square(comm, N, N)
    V = fem.functionspace(msh, ("Lagrange", k))
    x = ufl.SpatialCoordinate(msh)
    u_ex = x[0] * x[1] + ufl.sin(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1])
    f = -ufl.div(kappa(u_ex) * ufl.grad(u_ex))
    g = fem.Function(V); g.interpolate(fem.Expression(u_ex, V.element.interpolation_points))
    return msh, V, u_ex, f, fem.dirichletbc(g, boundary_dofs(msh, V))

def hw_solve(N, k, opts=TIGHT):
    """SNES from u0 = g on the boundary, 0 inside. Returns msh, uh, u_ex, f, bc, reason, its, history."""
    msh, V, u_ex, f, bc = hw_problem(N, k)
    uh = fem.Function(V); v = ufl.TestFunction(V)
    fem.set_bc(uh.x.array, [bc])
    F = kappa(uh) * ufl.inner(ufl.grad(uh), ufl.grad(v)) * ufl.dx - f * v * ufl.dx
    reason, its, hist = snes_solve(F, uh, [bc], opts)
    return msh, uh, u_ex, f, bc, reason, its, hist

> *your answer to (d)*

## Q3 — The one-dimensional Bratu problem (20 pts)

$-u''=\lambda e^u$ on $(0,1)$, $u(0)=u(1)=0$ (`mesh.create_unit_interval`). Compare everything with your exact values from Q1(c).

**(a) (6) Natural continuation.** P1, $N=64$, one `NonlinearProblem` with $\lambda$ a `fem.Constant`, $\lambda=0.5,1.0,\dots,3.5$ and then $3.6$.
Print reason, iterations and $\max u_h$. Where does it fail, with which reason, and why? Compare $\max u_h$ at $\lambda=2$ with the exact lower solution.

**(b) (8) The fold.** Copy `bratu_branch` and `fold` from lab §4 (they work unchanged on an interval mesh). For P1, $N=16,32,64,128$ compute $\lambda^*_h$
with `np.linspace(0.01, 1.2, 120)`, the error against the exact $\lambda^*$ and the observed order. Compare $\max u_h$ at the fold with $u^*(\frac12)$.

**(c) (6) The upper solution.** Run the branch further (`np.linspace(0.01, 2.0, 200)`, P1, $N=64$) and interpolate $\max u_h$ on the **upper** branch at
$\lambda=2$ (`np.interp` on the part after the fold, sorted by $\lambda$). Compare with the exact upper $u(\frac12)$. Why can natural continuation in
$\lambda$ never reach this solution from $\lambda=0$?

In [ ]:
%%fenicsx -np 2
# TODO: Q3(a), -np 2

from mpi4py import MPI
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem, NonlinearProblem

comm = MPI.COMM_WORLD
LU = {"ksp_type": "preonly", "pc_type": "lu"}
TIGHT = {**LU, "snes_rtol": 1e-10, "snes_atol": 1e-12, "snes_max_it": 50}

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def boundary_dofs(msh, V):
    tdim = msh.topology.dim
    msh.topology.create_connectivity(tdim - 1, tdim)
    return fem.locate_dofs_topological(V, tdim - 1, mesh.exterior_facet_indices(msh.topology))

def norm_L2(msh, e):
    return np.sqrt(msh.comm.allreduce(fem.assemble_scalar(fem.form(e ** 2 * ufl.dx)), op=MPI.SUM))

_n = [0]
def snes_solve(F, uh, bcs, opts):
    """Solve F(uh; v) = 0 with PETSc SNES. Returns (reason, iterations, history of ||F||)."""
    _n[0] += 1
    problem = NonlinearProblem(F, uh, bcs=bcs, petsc_options_prefix=f"nls{_n[0]}_", petsc_options=opts)
    hist = []
    problem.solver.setMonitor(lambda snes, it, rnorm: hist.append(rnorm))
    problem.solve()
    return problem.solver.getConvergedReason(), problem.solver.getIterationNumber(), hist

In [ ]:
%%fenicsx -np 2
# TODO: Q3(b) the fold for N = 16, 32, 64, 128, -np 2

from mpi4py import MPI
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem, NonlinearProblem

comm = MPI.COMM_WORLD
LU = {"ksp_type": "preonly", "pc_type": "lu"}
TIGHT = {**LU, "snes_rtol": 1e-10, "snes_atol": 1e-12, "snes_max_it": 50}

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def boundary_dofs(msh, V):
    tdim = msh.topology.dim
    msh.topology.create_connectivity(tdim - 1, tdim)
    return fem.locate_dofs_topological(V, tdim - 1, mesh.exterior_facet_indices(msh.topology))

def norm_L2(msh, e):
    return np.sqrt(msh.comm.allreduce(fem.assemble_scalar(fem.form(e ** 2 * ufl.dx)), op=MPI.SUM))

_n = [0]
def snes_solve(F, uh, bcs, opts):
    """Solve F(uh; v) = 0 with PETSc SNES. Returns (reason, iterations, history of ||F||)."""
    _n[0] += 1
    problem = NonlinearProblem(F, uh, bcs=bcs, petsc_options_prefix=f"nls{_n[0]}_", petsc_options=opts)
    hist = []
    problem.solver.setMonitor(lambda snes, it, rnorm: hist.append(rnorm))
    problem.solve()
    return problem.solver.getConvergedReason(), problem.solver.getIterationNumber(), hist

In [ ]:
%%fenicsx -np 2
# TODO: Q3(c) the upper solution at lam = 2, -np 2

from mpi4py import MPI
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem, NonlinearProblem

comm = MPI.COMM_WORLD
LU = {"ksp_type": "preonly", "pc_type": "lu"}
TIGHT = {**LU, "snes_rtol": 1e-10, "snes_atol": 1e-12, "snes_max_it": 50}

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def boundary_dofs(msh, V):
    tdim = msh.topology.dim
    msh.topology.create_connectivity(tdim - 1, tdim)
    return fem.locate_dofs_topological(V, tdim - 1, mesh.exterior_facet_indices(msh.topology))

def norm_L2(msh, e):
    return np.sqrt(msh.comm.allreduce(fem.assemble_scalar(fem.form(e ** 2 * ufl.dx)), op=MPI.SUM))

_n = [0]
def snes_solve(F, uh, bcs, opts):
    """Solve F(uh; v) = 0 with PETSc SNES. Returns (reason, iterations, history of ||F||)."""
    _n[0] += 1
    problem = NonlinearProblem(F, uh, bcs=bcs, petsc_options_prefix=f"nls{_n[0]}_", petsc_options=opts)
    hist = []
    problem.solver.setMonitor(lambda snes, it, rnorm: hist.append(rnorm))
    problem.solve()
    return problem.solver.getConvergedReason(), problem.solver.getIterationNumber(), hist

> *your answers to (a)–(c)*

## Q4 — LLM debugging: a continuation script with four bugs (20 pts)

An LLM was asked for "natural continuation in $\lambda$ for $-\Delta u-\lambda e^u=f$ on the unit square with a manufactured solution
$\tilde u=\sin(\pi x)\sin(\pi y)$, $\lambda=1,2,3,4$, and an $L^2$ error check at the final $\lambda$ for $N=8,16,32$". It produced the code below.
It runs without an error message with `-np 2`.

```python
from mpi4py import MPI
import numpy as np, ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import NonlinearProblem

def run(N):
    msh = mesh.create_unit_square(MPI.COMM_WORLD, N, N)
    V = fem.functionspace(msh, ("Lagrange", 1))
    x = ufl.SpatialCoordinate(msh)
    lam = 1.0
    u_ex = ufl.sin(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1])
    f = -ufl.div(ufl.grad(u_ex)) - lam * ufl.exp(u_ex)
    uh, v, du = fem.Function(V), ufl.TestFunction(V), ufl.TrialFunction(V)
    F = ufl.inner(ufl.grad(uh), ufl.grad(v)) * ufl.dx - lam * ufl.exp(uh) * v * ufl.dx - f * v * ufl.dx
    J = ufl.inner(ufl.grad(du), ufl.grad(v)) * ufl.dx - lam * du * v * ufl.dx
    dofs = fem.locate_dofs_geometrical(V, lambda X: np.isclose(X[0], 0) | np.isclose(X[0], 1) | np.isclose(X[1], 0) | np.isclose(X[1], 1))
    problem = NonlinearProblem(F, uh, J=J, bcs=[fem.dirichletbc(0.0, dofs, V)], petsc_options_prefix=f"llm{N}_",
                               petsc_options={"snes_max_it": 5, "snes_rtol": 1e-10, "ksp_type": "preonly", "pc_type": "lu"})
    for lam in (1.0, 2.0, 3.0, 4.0):
        problem.solve()
    u_I = fem.Function(V); u_I.interpolate(fem.Expression(u_ex, V.element.interpolation_points))
    return np.sqrt(np.mean((uh.x.array - u_I.x.array) ** 2))

for N in (8, 16, 32):
    print(N, run(N))
```

**(a) (8)** Run it unchanged (next cell, `-np 2`) and find the **four** bugs. For each: the line, what is wrong, the symptom, and **how you detected it**
(the output alone looks plausible — what did you print or test?).
**(b) (8)** Write the corrected version: print the reason and iterations after every solve, and the $L^2$ error at $\lambda=4$ with its rate.
**(c) (4)** Which of the four bugs would the final MMS table **not** reveal on its own? What does that say about MMS as the only check?

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import numpy as np, ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import NonlinearProblem

def run(N):
    msh = mesh.create_unit_square(MPI.COMM_WORLD, N, N)
    V = fem.functionspace(msh, ("Lagrange", 1))
    x = ufl.SpatialCoordinate(msh)
    lam = 1.0
    u_ex = ufl.sin(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1])
    f = -ufl.div(ufl.grad(u_ex)) - lam * ufl.exp(u_ex)
    uh, v, du = fem.Function(V), ufl.TestFunction(V), ufl.TrialFunction(V)
    F = ufl.inner(ufl.grad(uh), ufl.grad(v)) * ufl.dx - lam * ufl.exp(uh) * v * ufl.dx - f * v * ufl.dx
    J = ufl.inner(ufl.grad(du), ufl.grad(v)) * ufl.dx - lam * du * v * ufl.dx
    dofs = fem.locate_dofs_geometrical(V, lambda X: np.isclose(X[0], 0) | np.isclose(X[0], 1) | np.isclose(X[1], 0) | np.isclose(X[1], 1))
    problem = NonlinearProblem(F, uh, J=J, bcs=[fem.dirichletbc(0.0, dofs, V)], petsc_options_prefix=f"llm{N}_",
                               petsc_options={"snes_max_it": 5, "snes_rtol": 1e-10, "ksp_type": "preonly", "pc_type": "lu"})
    for lam in (1.0, 2.0, 3.0, 4.0):
        problem.solve()
    u_I = fem.Function(V); u_I.interpolate(fem.Expression(u_ex, V.element.interpolation_points))
    return np.sqrt(np.mean((uh.x.array - u_I.x.array) ** 2))

for N in (8, 16, 32):
    print(N, run(N))

In [ ]:
%%fenicsx -np 2
# TODO: Q4 diagnostics — print what you need to expose each bug, -np 2

> *your table for (a)*

In [ ]:
%%fenicsx -np 2
# TODO: Q4(b) corrected script, -np 2

> *your answer to (c)*

## 🤖 AI-usage log

*Required in every assignment and project. Using an LLM is encouraged and never penalized — an empty or vague log is.*

| # | Tool / model | What I asked (key prompt, shortened) | What it got wrong or missed | How I detected it | How I fixed it |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

**Code provenance** — which cells/functions were mostly generated, which did you write or substantially change?

> …

**Verification** — how do you know the final result is correct (MMS rates, benchmark value, conservation check, parallel = serial)?

> …

**Reflection (2–3 sentences)** — what would you do differently when prompting next time?

> …